# Spam Detection for Beginners

Have you ever received a text message saying that you won a free prize, or asking you to call a number right away? Messages like that are called **spam**. Spam means unwanted junk messages.

This notebook is a friendly guide that goes one small step at a time. Together we will teach a computer to read a message and decide if it is spam or a normal message. A normal message is often called **ham**, so from now on ham means not spam.

Here is the plan:

1. Load a real file of text messages that people already marked as spam or ham.
2. Explore the data and clean it.
3. Turn the text into numbers that a model can work with.
4. Train a multinomial Naive Bayes model.
5. Measure how good the model is.
6. Use the model to check our own messages.

You do not need any machine learning knowledge before you start. Run the cells from top to bottom. To run a cell, click on it and then press Shift and Enter together.

## Step 1: Bring in the tools we need

A library is a box of ready made tools that other people already wrote for us. We only need two of them.

**pandas** works with tables of data. It opens our file and lets us look at rows and columns, a bit like a spreadsheet, and it is also handy for cleaning the text.

**scikit-learn** is the machine learning box. It turns text into numbers, trains the model, and measures how good the model is.

That is all. Nothing needs to be installed.

The next cell brings both boxes in.

In [1]:
# pandas is the tool for tables of data.
import pandas as pd

# TfidfVectorizer turns text into numbers.
from sklearn.feature_extraction.text import TfidfVectorizer

# train_test_split cuts the data into a training part and a testing part.
from sklearn.model_selection import train_test_split

# MultinomialNB is the Naive Bayes model for text.
from sklearn.naive_bayes import MultinomialNB

# These three tools measure how well the model did.
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

## Step 2: Load the dataset

A dataset is a file full of examples for the computer to learn from.

Our file is called spam.csv and it sits inside the dataset folder of this project. The messages inside are real text messages, and each one is already marked as spam or ham.

One important detail: the file is stored in an old text format called latin-1. If we do not tell pandas about this, pandas stops with an error message. That is why we add encoding, with the value latin-1, when we read the file.

The next cell reads the file into a variable named data and then shows the first five rows.

In [2]:
# Read the file into a table named data.
# encoding tells pandas that the file uses the latin-1 text format.
data = pd.read_csv("dataset/spam.csv", encoding="latin-1")

# Look at the first five rows to be sure the data arrived well.
data.head()

,v1,v2,Unnamed: 2,Unnamed: 3,Unnamed: 4
0,ham,"Go until jurong point, crazy.. Available only ...",NaN,NaN,NaN
1,ham,Ok lar... Joking wif u oni...,NaN,NaN,NaN
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,NaN,NaN,NaN
3,ham,U dun say so early hor... U c already then say...,NaN,NaN,NaN
4,ham,"Nah I don't think he goes to usf, he lives aro...",NaN,NaN,NaN


## Step 3: Explore the data

Before we train anything, it is wise to look at what we have. We want to know three things:

* how many messages there are,
* how many of them are spam and how many are ham,
* and what the messages really look like.

The next cells answer those questions. First we print the shape, which tells us the number of rows and the number of columns. Then we count the labels. Then we print a few real messages of each kind.

In [3]:
# The shape gives us the number of rows and the number of columns.
print("Rows and columns:", data.shape)

Rows and columns: (5572, 5)


In [4]:
# Count how many messages we have of each kind.
data["v1"].value_counts()

v1
ham     4825
spam     747
Name: count, dtype: int64

In [5]:
# Print three normal messages so we can see what ham looks like.
print("Examples of ham messages:")

# The column v2 holds the message text.
print(data[data["v1"] == "ham"]["v2"].head(3).to_list())

Examples of ham messages:
['Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...', 'Ok lar... Joking wif u oni...', 'U dun say so early hor... U c already then say...']


In [6]:
# Print three spam messages so we can see what spam looks like.
print("Examples of spam messages:")
print(data[data["v1"] == "spam"]["v2"].head(3).to_list())

Examples of spam messages:
["Free entry in 2 a wkly comp to win FA Cup final tkts 21st May 2005. Text FA to 87121 to receive entry question(std txt rate)T&C's apply 08452810075over18's", "FreeMsg Hey there darling it's been 3 week's now and no word back! I'd like some fun you up for it still? Tb ok! XxX std chgs to send, å£1.50 to rcv", 'WINNER!! As a valued network customer you have been selected to receivea å£900 prize reward! To claim call 09061701461. Claim code KL341. Valid 12 hours only.']


## Step 4: Clean the data

Raw data is usually a little messy, so we tidy it up first. Cleaning here means four simple things:

1. Keep only the two columns we need. The column named v1 holds the label, which is ham or spam. The column named v2 holds the message text. The other three columns are empty, so we throw them away.
2. Give those two columns easy names: label and message.
3. Remove rows that are empty, and rows that are exact copies of each other. Copies would make the model think a message is more common than it really is.
4. Turn the words into numbers, because a model can only do maths with numbers. We use 0 for ham and 1 for spam.

After all that we count the labels again, so we can see how many rows are left.

In [7]:
# Keep only the label column and the message column.
data = data[["v1", "v2"]]

# Rename them to label and message, which is much easier to read.
data.columns = ["label", "message"]

# Show how many rows we have now.
print("Rows after keeping two columns:", len(data))

Rows after keeping two columns: 5572


In [8]:
# Remove rows where the label or the message is missing.
data = data.dropna()

# Remove spaces at the start and at the end of each message.
data["message"] = data["message"].astype(str).str.strip()

# Remove rows where the message became empty.
data = data[data["message"] != ""]

# Remove rows that are exact copies of each other.
data = data.drop_duplicates(subset=["message"])

# Show how many rows are left after this cleaning.
print("Rows after removing empty rows and copies:", len(data))

Rows after removing empty rows and copies: 5158


In [9]:
# Change the words ham and spam into the numbers 0 and 1.
data["label"] = data["label"].astype(str).str.strip().str.lower().map({"ham": 0, "spam": 1})

# Count the labels again to see how many we have of each.
print(data["label"].value_counts())

label
0    4516
1     642
Name: count, dtype: int64


## Step 5: Clean the text inside the messages

Next we tidy up the messages themselves. The model treats every different word as a clue, so tidier text means better clues. We do three small things:

* Make everything lowercase, so the words Free and free count as the same word instead of two different ones.
* Replace symbols such as dots, commas, and exclamation marks with a space.
* Squeeze several spaces into a single space.

The next two cells show one message before cleaning and then the same message after cleaning, so you can see the difference.

In [10]:
# Save the first message as it looks right now, so we can compare it later.
original_message = data["message"].iloc[0]

# Print the message before cleaning.
print("Before cleaning:")
print(original_message)

Before cleaning:
Go until jurong point, crazy.. Available only in bugis n great world la e buffet... Cine there got amore wat...


In [11]:
# Make every letter lowercase.
data["message"] = data["message"].str.lower()

# Replace every symbol, such as a dot or an exclamation mark, with a space.
data["message"] = data["message"].str.replace(r"[^a-z0-9\s]", " ", regex=True)

# Squeeze several spaces into one single space.
data["message"] = data["message"].str.replace(r"\s+", " ", regex=True)

# Remove the space at the start and at the end.
data["message"] = data["message"].str.strip()

# Print the same message after cleaning.
print("After cleaning:")
print(data["message"].iloc[0])

After cleaning:
go until jurong point crazy available only in bugis n great world la e buffet cine there got amore wat


## Step 6: Split the data into study pages and exam pages

Why do we split the data? Imagine a student who studies exactly the questions that will be on the exam, and remembers the answers. A high mark proves nothing about real understanding.

It is the same with a model. If we train it on all the messages and then test it with the very same messages, it can simply remember them, and we would wrongly believe it is clever.

So we cut the messages into two piles:

* the **training data**, which the model learns from, and
* the **testing data**, which we hide from the model until the end.

We use 80 percent of the messages for training and the remaining 20 percent for testing. The setting random_state with the value 42 is only a fixed starting point, so that everybody who runs this notebook gets exactly the same split and the same scores.

The option stratify, with y inside, keeps the same mix of spam and ham in both piles.

In [12]:
# Put the messages in one variable and the correct answers in another.
x = data["message"]
y = data["label"]

# Keep 20 percent for testing and give the rest to training.
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)

# Show how many messages ended up in each pile.
print("Training messages:", len(x_train))
print("Testing messages:", len(x_test))

Training messages: 4126
Testing messages: 1032


## Step 7: Turn the text into numbers with TF-IDF

A model cannot read words. It can only do maths with numbers, so every message has to become a list of numbers.

TF-IDF is a clever way of counting words. It gives a high number to a word that is important in one message and rare in the others, such as the word winner, and a low number to a word that appears in almost every message, such as the word the. In short, TF-IDF turns each message into numbers, and the words that carry meaning stand out.

We also set stop_words, with english as the value, so that very common words such as the and at are ignored completely, because they carry almost no meaning.

Two calls are used on purpose. We fit on the training messages, which means the tool learns the words from the training pile only. Then we transform the testing messages with that same tool, so the model never peeks at the testing data.

In [13]:
# Create the tool that turns text into numbers.
vectorizer = TfidfVectorizer(stop_words="english")

# Learn the words from the training messages, then turn them into numbers.
x_train_numbers = vectorizer.fit_transform(x_train)

# Turn the testing messages into numbers with the very same words.
x_test_numbers = vectorizer.transform(x_test)

# Show the size of the training table: rows are messages and columns are words.
print("Number of training messages and words:", x_train_numbers.shape)

Number of training messages and words: (4126, 7289)


## Step 8: Train the model with Naive Bayes

Naive Bayes is the model we will use, and it is one of the simplest models for text.

Here is the idea in plain words. While training, the model reads every message and writes down how often each word shows up in spam messages and how often it shows up in ham messages. Words such as free, prize, and winner appear mostly in spam. Words such as homework and dinner appear mostly in ham.

When a new message arrives, the model looks at its words and asks one question: which group would be most likely to produce exactly these words? Then it gives its answer.

The word naive means that the model assumes each word is independent of the other words. Real language does not work like that, but the model still works very well in practice.

The setting alpha, with the value 0.1, is called smoothing. It stops the model from giving a probability of zero to a word it has never seen before. The value 0.1 works better than the usual starting value of 1.0 on this dataset.

In [14]:
# Create the Naive Bayes model. alpha is the smoothing value explained above.
model = MultinomialNB(alpha=0.1)

# Train the model: give it the training numbers together with the correct labels.
model.fit(x_train_numbers, y_train)

# Print a short message so we know the training finished.
print("Training finished")

Training finished


## Step 9: Test the model and understand the scores

Now we let the model guess the labels of the testing messages, which it has never seen. Then we compare its guesses with the real answers.

Three words describe the result:

* **Accuracy** is the share of all testing messages that were guessed correctly.
* **Precision**, for the spam label, answers this question: of all the messages the model called spam, how many really were spam? A low precision means the model accuses innocent messages.
* **Recall**, for the spam label, answers this question: of all the real spam messages, how many did the model manage to catch? A low recall means real spam slips through the net.

An easy way to remember them: precision is about being sure before you accuse, and recall is about not letting the guilty ones escape.

In [15]:
# Ask the model to guess the label of every testing message.
guesses = model.predict(x_test_numbers)

# Compare the guesses with the real answers and print the accuracy.
print("Accuracy:", accuracy_score(y_test, guesses))

Accuracy: 0.9854651162790697


In [16]:
# Print a report with precision, recall, and f1 for each label.
# The support column shows how many messages of each label are in the testing pile.
print(classification_report(y_test, guesses))

              precision    recall  f1-score   support

           0       0.99      1.00      0.99       904
           1       0.97      0.91      0.94       128

    accuracy                           0.99      1032
   macro avg       0.98      0.95      0.97      1032
weighted avg       0.99      0.99      0.99      1032



## Step 10: What those numbers mean for our model

Our run printed an accuracy of about 0.99, which is 99 percent. In plain words, out of 1032 testing messages the model guessed about 1017 correctly and got about 15 wrong. That is a strong result.

Look at the spam row of the report:

* Precision is 0.97. So when the model says spam, it is right about 97 times out of 100. Very few normal messages were wrongly called spam.
* Recall is 0.91. So the model caught about 91 spam messages out of every 100, and let around 9 out of every 100 through. The misses are the price we pay for such a simple model.

Now look at the ham row. Precision is 0.99 and recall is 1.00, which means normal messages were almost never disturbed.

One last thing worth noticing is the support column. It shows that the testing pile had about 904 ham messages but only about 128 spam messages. Spam is the rare case here, and that is exactly why we also read recall: a model can look very accurate while still waving some spam through the door.

## Step 11: The confusion matrix

A confusion matrix is a small table that shows the four possible outcomes of a guess. Reading it is easy if you remember two pairs of words:

* **True** means the model was right, and **false** means the model was wrong.
* **Not Spam** means the model believed the message was fine, and **Spam** means the model believed the message was junk.

Put together, the four boxes of the table mean:

* True Spam, meaning the message really was spam and the model said spam. A good catch.
* False Spam, meaning the message was normal but the model said spam. A false alarm.
* False Not Spam, meaning the message was spam but the model said normal. A message that slipped through.
* True Not Spam, meaning the message was normal and the model said normal. A correct and quiet outcome.

The next cell builds that table with clear names on the rows and the columns, so it reads like a small report instead of a block of numbers.

In [17]:
# Build a small table from the confusion matrix, with clear row and column names.
results_table = pd.DataFrame(
    confusion_matrix(y_test, guesses),
    index=["Real: Not Spam", "Real: Spam"],
    columns=["Model said: Not Spam", "Model said: Spam"],
)

# Show the table.
results_table

,Model said: Not Spam,Model said: Spam
Real: Not Spam,900,4
Real: Spam,11,117


## Step 12: How to read our table

Our table came out like this.

* Top left, 900: normal messages that the model correctly called normal.
* Top right, 4: normal messages that the model wrongly called spam. Only 4 false alarms among 904 normal messages.
* Bottom left, 11: real spam messages that the model let through, and these are the misses we would like to reduce.
* Bottom right, 117: spam messages that the model caught correctly.

Two quick checks show that the table agrees with the score above. Add all four boxes, and you get 1032, which is exactly the number of testing messages. Add the two correct boxes, 900 and 117, and you get 1017 right answers, which is the accuracy of about 99 percent that we printed in Step 9.

In short: the model is very careful with normal messages and catches most spam, and the small box of 11 tells us where the remaining weakness lies.

## Step 13: A function to check any message

Now we wrap the whole prediction into one small function. A function is a block of code with a name, so we can use it again and again instead of repeating ourselves.

Our function is called check_message and it does four things:

1. It cleans the new text in exactly the same way we cleaned the training messages. This matters a lot, because the model must see a new message in the same shape as the messages it learned from.
2. It turns the cleaned text into numbers with the same vectorizer that we trained in Step 7.
3. It asks the model to guess, which gives 0 for ham and 1 for spam.
4. It prints the answer together with a confidence, which is how sure the model is about its own guess. A confidence of 99 percent means the model is very sure.

The first cell below defines a tiny helper for cleaning one message, because a new message arrives as a single piece of text and not as a whole table.

In [18]:
# Define a small function that cleans one single message.
def clean_text(text):
    # Put the one message into a tiny pandas table, so we can use the same tools.
    small = pd.Series([text])

    # Make it lowercase, exactly as we did for the training messages.
    small = small.str.lower()

    # Replace symbols with a space, exactly as before.
    small = small.str.replace(r"[^a-z0-9\s]", " ", regex=True)

    # Squeeze extra spaces, also exactly as before.
    small = small.str.replace(r"\s+", " ", regex=True)

    # Give back the cleaned message as one ordinary piece of text.
    return small.str.strip().iloc[0]

In [19]:
# Check that the cleaning helper does what we expect on one loud example.
# Notice how the exclamation marks disappear and the text turns lowercase.
print(clean_text("WINNER!! You have won a FREE prize NOW!!!"))

winner you have won a free prize now


In [20]:
# Define the function that checks one message and prints the answer.
def check_message(message):
    # Step 1: clean the message with the same steps used during training.
    cleaned = clean_text(message)

    # Step 2: turn the cleaned text into numbers with our trained vectorizer.
    numbers = vectorizer.transform([cleaned])

    # Step 3: let the model guess. The guess is 0 for ham and 1 for spam.
    guess = model.predict(numbers)[0]

    # Also ask the model how sure it is about that guess, as a percentage.
    confidence = model.predict_proba(numbers)[0][guess] * 100

    # Step 4: choose a friendly word for the answer.
    if guess == 1:
        answer = "Spam"
    else:
        answer = "Not Spam"

    # Print the answer together with the confidence, rounded to one decimal place.
    print("Message:", message)
    print("Answer:", answer, "with a confidence of", round(confidence, 1), "percent")
    print("-" * 60)

## Step 14: Try it on a few messages

Let us test the model with four messages: two that look like spam and two that look normal. Watch three things while you read the output: whether the answer makes sense to you, how high the confidence is, and how a short and clear message compares with a long one.

The next two cells handle the spam examples, and the two cells after them handle the normal ones.

In [21]:
# Spam example 1: a message about winning a prize and calling a number.
check_message("WINNER! You have won a 1000 pound prize. Call 09061701461 to claim your reward now.")

Message: WINNER! You have won a 1000 pound prize. Call 09061701461 to claim your reward now.
Answer: Spam with a confidence of 100.0 percent
------------------------------------------------------------


In [22]:
# Spam example 2: a competition message that asks you to send a text message.
check_message("Free entry in 2 a weekly competition to win FA Cup final tickets. Text WIN to 87121 now.")

Message: Free entry in 2 a weekly competition to win FA Cup final tickets. Text WIN to 87121 now.
Answer: Spam with a confidence of 99.5 percent
------------------------------------------------------------


In [23]:
# Normal example 1: a friendly message about lunch.
check_message("Hey, are we still meeting for lunch at 1 pm tomorrow?")

Message: Hey, are we still meeting for lunch at 1 pm tomorrow?
Answer: Not Spam with a confidence of 99.8 percent
------------------------------------------------------------


In [24]:
# Normal example 2: a short message about homework.
check_message("I finished the homework. I will send you the notes tonight.")

Message: I finished the homework. I will send you the notes tonight.
Answer: Not Spam with a confidence of 99.2 percent
------------------------------------------------------------


## Step 15: Now it is your turn

Change the text between the quotes in the next cell to anything you like, and then run the cell again. The model will give you its answer and its confidence.

A good experiment is to write two messages that say almost the same thing but in different words, and see if the confidence changes. You can also try a message that mixes both styles, such as a normal sentence that happens to contain the word free.

In [25]:
# Write your own message here. Keep the quotes around the text.
my_message = "Congratulations, you have won a free ticket. Reply YES to claim it now."

# Ask the model to check your message.
check_message(my_message)

Message: Congratulations, you have won a free ticket. Reply YES to claim it now.
Answer: Spam with a confidence of 97.4 percent
------------------------------------------------------------


## What we learned, and what to try next

Well done. You built a working spam filter from scratch and you followed every step.

In this notebook we:

* loaded a real file of text messages with pandas, and dealt with its latin-1 encoding,
* removed empty rows and copies, and turned ham and spam into 0 and 1,
* cleaned the text by making it lowercase and removing symbols,
* split the messages into a training pile and a testing pile, and learned why that matters,
* turned the text into numbers with TF-IDF,
* trained a multinomial Naive Bayes model with alpha equal to 0.1,
* judged the model with accuracy, precision, and recall, and read a confusion matrix,
* and packed everything into one small function that checks any message.

Ideas for next steps:

1. Try to catch more spam. In our table 11 spam messages slipped through, so try other values of alpha, such as 0.05 and 0.5, and compare the recall of the spam row each time.
2. Collect your own mistakes. Run the model on messages you receive, note the ones it gets wrong, and think about which words fooled it. Reading mistakes teaches you more than reading a score.
3. Compare with a simpler way of counting words, using plain word counts instead of TF-IDF, and see how much TF-IDF really helps.
4. Remember the limit of the model. It only knows the words it saw during training, so a brand new kind of spam with new words can still slip through. A spam filter is a helper, not a guarantee.

The same cleaning steps and the same model are used by the small Streamlit app in this project, so now you also understand what that app does under the hood.